Входные данные, в будущем лучше грузить из файла

In [1]:
import numpy as np
import pandas as pd
import math
# import matplotlib.pyplot as plt

In [2]:
# My library imports
import fluid
from function_dp import vniigaz_vertical_one, vniigaz_vertical_two, vniigaz_inclined, vniigaz_horizontal

In [4]:
lyamda=0.0140
diameter=0.1000
alfa=85
v=1.95E-03
u=5.643
Ro_water=1000.00
Ro_gas=19.24764133
pressure=2.896463361
sigma=0.072
q_liquid=1.52778E-05
q_gas=1.273148148

# Тест в первой функции
horizont = vniigaz_horizontal(lyamda, diameter, alfa, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid, q_gas)
print("dp_dz from vniigaz_horizont:", horizont)

inclined = vniigaz_inclined(lyamda, diameter, alfa, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid, q_gas)
print("dp_dz from vniigaz_inclined:", inclined)

vertical = vniigaz_vertical_one(lyamda, diameter, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid, q_gas)
print("dp_dz from vniigaz_vertical_one:", vertical)

vertical_two = vniigaz_vertical_two(lyamda, diameter, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid, q_gas)
print("dp_dz from vniigaz_two:", vertical_two)

# # Тест через класс (если он у тебя ещё остался)
# mod = vniigaz_two(lamda_tube, diameter)
# dp2 = mod.get_dp_dz(v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid, q_gas)
# print("dp_dz from class:", dp2)

# # Функция для горизонтального участка
# dp2_1 = vniigaz_horizontal_dp_dz(lamda_tube, diameter, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid, q_gas, alfa)
# print("dp_dz from vniigaz_horizontal_dp_dz:", dp2_1)

# # Тест через класс для горизонтального участка
# mod_hor = vniigaz_horizontal(lamda_tube, diameter)
# dp2_1 = mod_hor.get_dp_dz(v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid, q_gas, alfa)
# print("dp_dz from horizontal class:", dp2_1)


dp_dz from vniigaz_horizont: 59.3548618791328
0.005592589756640183
dp_dz from vniigaz_inclined: 71.29564226173795
dp_dz from vniigaz_vertical_one: 309.44971462708656
dp_dz from vniigaz_two: 269.8371185054112


In [5]:
# Standard conditions
Pstd = 101325 #Pa
Tstd = 273.15 + 20 #K

# Fluid
Ro_gas_std = 0.67 # кг/м3
Ro_water_std = 1000 # кг/м3
sigma = 0.072 # Н/м, поверхностное натяжение

# Reservoir
permability = 10*10**-12 # м2
pay_thickness = 10 # м

reservoir_temp = 305.15 # K
wellhead_temp = 293.15 # K
reservoir_pressure = 2.89646336072114 # МПа

# Data for gas-liquid flow
Q_gas_std = 100 # тыс.м3/сутки
diameter = 0.1 # м, диаметр трубы
lyamda = 0.02
wgr = 12*10**-5 # м3/м3, водогазовый фактор


In [3]:
# Well properties
# Download inclinometry data
file_path_incl = 'd:/Python_2025/old_rep/исходные_данные/17.dev'  # Update this path to match your actual file location
with open(file_path_incl, 'r', encoding='utf-8') as inclinometry_file:
    well_data = pd.read_csv(inclinometry_file, comment='#', sep=r'\s+')

# Download inclinometry data_horizontal
file_path_incl_horizontal = 'd:/Python_2025/old_rep/исходные_данные/10102.dev'  # Update this path to match your actual file location
with open(file_path_incl_horizontal, 'r', encoding='utf-8') as inclinometry_file_horizontal:
    well_data_hor = pd.read_csv(inclinometry_file_horizontal, comment='#', sep=r'\s+')

# Gas properties
# Download PVT data
file_path_pvt = 'd:/Python_2025/old_rep/исходные_данные/PVT.inc'  # Update this path to match your actual file location
with open(file_path_pvt, 'r', encoding='utf-8') as pvt_file:
    pvt_data = pd.read_csv(pvt_file, comment='#', sep=r'\s+')

# Test fuid properties
# Create a fluid object with the given properties
# rho_c=0.6799, xa=0.8858, xy=0.0668
Ro_gas_std = 0.6799
xa = 0.8858
xy = 0.0668
fluid = fluid.Fluid(Ro_gas_std, xa, xy, pvt_data)

In [9]:
# 3300.00000 last md
well_propertis = pd.DataFrame({
    'MD': well_data_hor['MD'],
    'TVD': well_data_hor['TVD'],
    'alfa': well_data_hor['INCL'],
    'Diameter': diameter,
    'T': np.nan,
    'Pi': np.nan,
    'Pi1': np.nan,
    'z': np.nan,
    'Ro_gas': np.nan,
    'u': np.nan,
    'v': np.nan,
    'Qgas_pt': np.nan,
    'Dp_dz': np.nan
})



In [10]:
for idx in well_propertis.index[::-1]:

    # Заполнение параметра Т в зависимости от глубины скважины
    if idx == 0:
        well_propertis.loc[idx, 'T'] = wellhead_temp
    elif idx == len(well_propertis)-1:
        well_propertis.loc[idx, 'T'] = reservoir_temp
    else:
        well_propertis.loc[idx, 'T'] = reservoir_temp - (reservoir_temp - wellhead_temp) * (well_propertis.loc[idx, 'TVD'] / well_propertis['TVD'].iloc[-1])

    # Заполнение параметра Pi в зависимости от глубины скважины
    if idx == len(well_propertis)-1:
        well_propertis.loc[idx, 'Pi'] = reservoir_pressure

    tempreture = well_propertis.loc[idx, 'T'] # К
    pressure = well_propertis.loc[idx, 'Pi'] # МПа

    # Расчёт z
    if not pd.isna(well_propertis.loc[idx, 'Pi']):
        well_propertis.loc[idx, 'z'] = fluid.get_Z(pressure, tempreture)
        well_propertis.loc[idx, 'Ro_gas'] = Ro_gas_std * pressure/tempreture/fluid.get_Z(pressure, tempreture)*Tstd*1000000/Pstd
        well_propertis.loc[idx, 'u'] = (Q_gas_std*1000/86400) * Ro_gas_std / well_propertis.loc[idx, 'Ro_gas'] / (np.pi*well_propertis.loc[idx, 'Diameter']**2/4)
        well_propertis.loc[idx, 'v'] = ((Q_gas_std*1000/86400)*wgr)/(np.pi*well_propertis.loc[idx, 'Diameter']**2/4)
        well_propertis.loc[idx, 'Qgas_pt'] = (Q_gas_std*1000/86400) * fluid.get_Bg(pressure, tempreture)
        if well_propertis.loc[idx, 'alfa'] < 85:
            well_propertis.loc[idx, 'Dp_dz'] = vniigaz_inclined(lyamda, diameter, well_propertis.loc[idx, 'alfa'], well_propertis.loc[idx, 'v'], well_propertis.loc[idx, 'u'], Ro_water_std, well_propertis.loc[idx, 'Ro_gas'], pressure, sigma, ((Q_gas_std*1000/86400)*wgr), well_propertis.loc[idx, 'Qgas_pt'])
        else:
            well_propertis.loc[idx, 'Dp_dz'] = vniigaz_horizontal(lyamda, diameter, well_propertis.loc[idx, 'alfa'], well_propertis.loc[idx, 'v'], well_propertis.loc[idx, 'u'], Ro_water_std, well_propertis.loc[idx, 'Ro_gas'], pressure, sigma, ((Q_gas_std*1000/86400)*wgr), well_propertis.loc[idx, 'Qgas_pt'])
        if idx > 0:
            well_propertis.loc[idx, 'Pi1'] = well_propertis.loc[idx, 'Pi'] - (well_propertis.loc[idx, 'Dp_dz'] * (well_propertis.loc[idx, 'MD'] - well_propertis.loc[idx-1, 'MD']))*10**-6     
            well_propertis.loc[idx-1, 'Pi'] = well_propertis.loc[idx, 'Pi1']

0.01104245168935939
0.010992407958098254
0.010877287337883778
0.010904493031875958
0.01090686935509892
0.010938805791091341
0.010980190040738299
0.010884214762458128
0.01102249599257655
0.011044042610273797
0.010964422898278085
0.010993789754452622
0.011518722248866273
0.011936952333995608
0.012273603049117026
0.01255948679549926
0.012870639836076007
0.013571980440813172
0.014006658725308051
0.014519467871698586
0.015146667231048766
0.015758591089239805
0.01647350208767515
0.016938325389075876
0.01758751912474068
0.018350547915821264
0.019267133451693258
0.020241672081240038
0.021169167558823813
0.02215722305349899
0.023113011539392873
0.02394214550928899
0.024693040473095368
0.025492681257791736
0.026466906092474932
0.027724209002316667
0.029125619322453913
0.030215921871482764
0.03130716897444723
0.03201713972073384
0.032455711965096896
0.03258545027695007
0.03236525314144719
0.032145481237702105
0.032019643439248076
0.032092287062964145
0.03229135289602764
0.032407733366254736
0.032

In [24]:
# code DeepSeek расчёт BHP сверху в низ
# Идём ОТ УСТЬЯ (первая точка, MD=0) К ЗАБОЮ (последняя точка)

wellhead_pressure = 0.5
Q_gas_std =  10000
wgr = 0.000001 # м3/м3
Pstd = 101325
wellhead_temp = 293.15

well_properties_bhp = pd.DataFrame({
    'MD': well_data_hor['MD'],
    'TVD': well_data_hor['TVD'],
    'alfa': well_data_hor['INCL'],
    'Diameter': diameter,
    'T': np.nan,
    'P': np.nan,           # Давление в текущей точке (начинаем с устья)
    'z': np.nan,
    'Ro_gas': np.nan,
    'u_gas': np.nan,       # скорость газа
    'u_liq': np.nan,       # скорость жидкости
    'Qgas_pt': np.nan,
    'Dp_dz': np.nan,
    'ret_vnii': ''
})

for idx in well_properties_bhp.index:
    
    # Заполнение температуры
    if idx == 0:
        well_properties_bhp.loc[idx, 'T'] = wellhead_temp
    elif idx == len(well_properties_bhp)-1:
        well_properties_bhp.loc[idx, 'T'] = reservoir_temp
    else:
        well_properties_bhp.loc[idx, 'T'] = reservoir_temp - (reservoir_temp - wellhead_temp) * (well_properties_bhp.loc[idx, 'TVD'] / well_properties_bhp['TVD'].iloc[-1])
    
    # Заполнение давления - начинаем с УСТЬЯ
    if idx == 0:
        well_properties_bhp.loc[idx, 'P'] = wellhead_pressure  # известное устьевое давление
    
    temperature = well_properties_bhp.loc[idx, 'T']  # K
    pressure = well_properties_bhp.loc[idx, 'P']     # МПа
    
    # Расчёт свойств в текущей точке
    if not pd.isna(well_properties_bhp.loc[idx, 'P']):
        well_properties_bhp.loc[idx, 'z'] = fluid.get_Z(pressure, temperature)
        well_properties_bhp.loc[idx, 'Ro_gas'] = Ro_gas_std * pressure / temperature / fluid.get_Z(pressure, temperature) * Tstd * 1000000 / Pstd
        # well_properties_bhp.loc[idx, 'u_gas'] = (Q_gas_std * 1000 / 86400) * Ro_gas_std / well_properties_bhp.loc[idx, 'Ro_gas'] / (np.pi * well_properties_bhp.loc[idx, 'Diameter']**2 / 4)
        well_properties_bhp.loc[idx, 'u_gas'] = (Q_gas_std / 86400) * Ro_gas_std / well_properties_bhp.loc[idx, 'Ro_gas'] / (np.pi * well_properties_bhp.loc[idx, 'Diameter']**2 / 4)
        well_properties_bhp.loc[idx, 'u_liq'] = ((Q_gas_std * 1000 / 86400) * wgr) / (np.pi * well_properties_bhp.loc[idx, 'Diameter']**2 / 4)
        well_properties_bhp.loc[idx, 'Qgas_pt'] = (Q_gas_std * 1000 / 86400) * fluid.get_Bg(pressure, temperature)
        
        # Расчёт градиента давления
        if well_properties_bhp.loc[idx, 'alfa'] < 85:
            well_properties_bhp.loc[idx, 'Dp_dz'] = vniigaz_inclined(
                lyamda, diameter, well_properties_bhp.loc[idx, 'alfa'],
                well_properties_bhp.loc[idx, 'u_liq'], well_properties_bhp.loc[idx, 'u_gas'],
                Ro_water_std, well_properties_bhp.loc[idx, 'Ro_gas'],
                pressure, sigma,  # pressure в МПа -> Па
                ((Q_gas_std / 86400) * wgr),
                well_properties_bhp.loc[idx, 'Qgas_pt']
            )
            well_properties_bhp.loc[idx, 'ret_vnii'] = (
                f"lyamda={lyamda:.4f}, "
                f"diameter={diameter:.4f}, "
                f"alfa={well_properties_bhp.loc[idx, 'alfa']:.2f}, "
                f"u_liq={well_properties_bhp.loc[idx, 'u_liq']:.6f}, "
                f"u_gas={well_properties_bhp.loc[idx, 'u_gas']:.6f}, "
                f"Ro_water_std={Ro_water_std:.2f}, "
                f"Ro_gas={well_properties_bhp.loc[idx, 'Ro_gas']:.4f}, "
                f"pressure={pressure:.2f}, "
                f"sigma={sigma:.4f}, "
                f"Q_water={((Q_gas_std / 86400) * wgr):.8f}, "
                f"Qgas_pt={well_properties_bhp.loc[idx, 'Qgas_pt']:.8f}"
                )
        else:
            well_properties_bhp.loc[idx, 'Dp_dz'] = vniigaz_horizontal(
                lyamda, diameter, well_properties_bhp.loc[idx, 'alfa'],
                well_properties_bhp.loc[idx, 'u_liq'], well_properties_bhp.loc[idx, 'u_gas'],
                Ro_water_std, well_properties_bhp.loc[idx, 'Ro_gas'],
                pressure, sigma,
                ((Q_gas_std / 86400) * wgr),
                well_properties_bhp.loc[idx, 'Qgas_pt']
            )

        # КЛЮЧЕВОЕ ОТЛИЧИЕ: ПРИБАВЛЯЕМ градиент (давление растёт с глубиной)
        if idx < len(well_properties_bhp) - 1:
            delta_MD = well_properties_bhp.loc[idx+1, 'MD'] - well_properties_bhp.loc[idx, 'MD']
            # Давление в следующей точке = текущее давление + dp_dz * ΔMD
            pressure_next = pressure + (well_properties_bhp.loc[idx, 'Dp_dz'] * delta_MD) * 1e-6  # Па -> МПа
            well_properties_bhp.loc[idx+1, 'P'] = pressure_next

# Сохраняем результаты
well_properties_bhp.to_csv("data_10102_BHP_from_THP.csv", index=False, encoding="utf-8")

0.42596071276143543
0.40486962116390773
0.43211778733608686
0.4594850738539632
0.4867187537509407
0.5135770908081505
0.5398393754001044
0.5653137304757228
0.5898421812867722
0.6133028978152996
0.6356098738173899
0.6567105567024355
0.676582061793537
0.6952266064857723
0.7126666622637433
0.7289402480151486
0.7440966808890994
0.7581928687944779
0.7712902393515182
0.783452282878079
0.794742660993453
0.805223801111523
0.8149559043222308
0.8239962802177949
0.8323989404540936
0.8402143906064484
0.8474895667212433
0.8542678766011282
0.8605893145723424
0.8664906220941229
0.8720054784902023
0.8771647054005642
0.8819964768438687
0.8865265271598918
0.8907783521291744
0.8947733994017453
0.8985312476303325
0.9020697758418146
0.9054053140735532
0.9085527875560688
0.9115258453424405
0.9143369780146896
0.9169976257104475
0.919518274165922
0.9219085431212243
0.924177266337686
0.9263325634287795
0.9283819035014171
0.9303321652761417
0.9321896897282286
0.9339603265842111
0.9356494789099749
0.9372621430929

d:\Python_2025\Gas_liquid_flow\fluid.py:98: RuntimeWarning: invalid value encountered in scalar power
  A2 = (A0 - (A0**2 - A1**3)**0.5)**(1/3)
d:\Python_2025\Gas_liquid_flow\fluid.py:98: RuntimeWarning: invalid value encountered in scalar power
  A2 = (A0 - (A0**2 - A1**3)**0.5)**(1/3)
d:\Python_2025\Gas_liquid_flow\fluid.py:98: RuntimeWarning: invalid value encountered in scalar power
  A2 = (A0 - (A0**2 - A1**3)**0.5)**(1/3)


In [ ]:
# ============================================================
# КОНСТАНТЫ И ПАРАМЕТРЫ
# ============================================================

# Standard conditions
Pstd = 101325  # Pa
Tstd = 273.15 + 20  # K

# Fluid properties
Ro_gas_std = 0.67  # кг/м3
Ro_water_std = 1000  # кг/м3
sigma = 0.072  # Н/м

# Well parameters
diameter = 0.1  # м
lyamda = 0.02

# Temperature profile
wellhead_temp = 293.15  # K
reservoir_temp = 305.15  # K

# Load inclinometry data
file_path_incl_horizontal = 'd:/Python_2025/Gas_liquid_flow/исходные_данные/vertical_well.dev'
with open(file_path_incl_horizontal, 'r', encoding='utf-8') as inclinometry_file_vertical:
    well_data_hor = pd.read_csv(inclinometry_file_vertical, comment='#', sep=r'\s+')

# Maximum TVD (глубина по вертикали) - bottom hole depth
max_TVD = well_data_hor['TVD'].iloc[-1]

# Load PVT data
file_path_pvt = 'd:/Python_2025/Gas_liquid_flow/исходные_данные/PVT.inc'
with open(file_path_pvt, 'r', encoding='utf-8') as pvt_file:
    pvt_data = pd.read_csv(pvt_file, comment='#', sep=r'\s+')

# Create fluid object
xa = 0.8858
xy = 0.0668
fluid_obj = fluid.Fluid(Ro_gas_std, xa, xy, pvt_data)

# ============================================================
# ВХОДНЫЕ ДАННЫЕ ДЛЯ VFP ТАБЛИЦЫ
# ============================================================

# Номер таблицы
table_number = 1

# Дебиты газа FLO (м3/сут) - в возрастающем порядке
gas_rates = np.array([30000, 50000, 100000, 200000])

# Устьевые давления THP (бар) - в возрастающем порядке
thp_bar = np.array([5, 30, 80])
thp_pa = thp_bar * 1e5

# Водогазовый фактор WFR (WGR) (м3/м3) - в возрастающем порядке
wgr_values = np.array([0.000001, 0.00001, 0.0001])

# GFR (OGR) - не используем, но нужно для формата
ogr_values = np.array([0.0])

# ALQ (GRAT) - не используем
grat_values = np.array([0.0])

# ============================================================
# ФУНКЦИЯ ДЛЯ РАСЧЁТА BHP ПО ИЗВЕСТНОМУ THP
# ============================================================

def calculate_bhp_from_thp(thp_pa, Q_gas_std_m3_per_day, wgr, well_data, 
                           diameter, lyamda, Ro_water_std, sigma, 
                           fluid_obj, wellhead_temp, reservoir_temp,
                           Pstd, Tstd, Ro_gas_std):
    """
    Расчёт забойного давления BHP по известному устьевому давлению THP.
    Возвращает BHP в барах.
    """
    
    # Конвертируем дебит газа в м3/с
    Q_gas_std = Q_gas_std_m3_per_day / 86400
    
    # Расход жидкости (воды) при стандартных условиях
    Q_water_std = Q_gas_std * wgr
    
    # Создаём копию данных для расчёта
    df = well_data.copy()
    
    # Заполняем температуру
    for idx in df.index:
        if idx == 0:
            df.loc[idx, 'T'] = wellhead_temp
        elif idx == len(df)-1:
            df.loc[idx, 'T'] = reservoir_temp
        else:
            df.loc[idx, 'T'] = reservoir_temp - (reservoir_temp - wellhead_temp) * (df.loc[idx, 'TVD'] / df['TVD'].iloc[-1])
    
    # Начальное давление на устье (первая точка)
    df.loc[0, 'P'] = thp_pa
    
    # Идём от устья к забою
    for idx in range(len(df)-1):
        pressure = df.loc[idx, 'P']
        temperature = df.loc[idx, 'T']
        alfa = df.loc[idx, 'INCL']
        
        # PVT свойства
        try:
            pressure_mpa = pressure / 1e6
            Bg = fluid_obj.get_Bg(pressure_mpa, temperature)
            if np.isnan(Bg) or np.isinf(Bg):
                Bg = 0.01
        except:
            Bg = 0.01
        
        # Плотность газа через Bg
        Ro_gas = Ro_gas_std / Bg
        
        # Расход газа при скважинных условиях
        Q_gas_pt = Q_gas_std * Bg
        
        # Площадь сечения
        area = np.pi * diameter**2 / 4
        
        # Скорости
        u_gas = Q_gas_pt / area if Q_gas_pt > 0 else 0
        
        if Q_water_std > 0:
            Q_water_pt = Q_water_std
            u_liq = Q_water_pt / area
        else:
            Q_water_pt = 0.0
            u_liq = 0.0
        
        # Расчёт градиента
        try:
            if alfa < 85:
                dp_dz = vniigaz_inclined(
                    lyamda, diameter, alfa,
                    u_liq, u_gas,
                    Ro_water_std, Ro_gas,
                    pressure, sigma,
                    Q_water_pt, Q_gas_pt
                )
            else:
                dp_dz = vniigaz_horizontal(
                    lyamda, diameter, alfa,
                    u_liq, u_gas,
                    Ro_water_std, Ro_gas,
                    pressure, sigma,
                    Q_water_pt, Q_gas_pt
                )
            
            if np.isnan(dp_dz) or np.isinf(dp_dz):
                dp_dz = 1000
        except:
            dp_dz = 1000
        
        # Переход к следующей точке
        delta_MD = df.loc[idx+1, 'MD'] - df.loc[idx, 'MD']
        pressure_next = pressure + dp_dz * delta_MD
        
        if np.isnan(pressure_next) or np.isinf(pressure_next):
            return 1e10
        
        df.loc[idx+1, 'P'] = pressure_next
        
        if pressure_next > 1e10:
            return 1e10
    
    # Возвращаем BHP в барах
    bhp_pa = df.loc[len(df)-1, 'P']
    bhp_bar = bhp_pa / 1e5
    
    return bhp_bar


# ============================================================
# РАСЧЁТ ВСЕХ КОМБИНАЦИЙ
# ============================================================

print("=" * 70)
print("ГЕНЕРАЦИЯ VFP ТАБЛИЦЫ")
print("=" * 70)
print(f"Таблица №{table_number}")
print(f"Глубина забоя: {max_TVD:.0f} м")
print(f"Дебиты газа (FLO): {gas_rates}")
print(f"Устьевые давления (THP): {thp_bar} бар")
print(f"Водогазовый фактор (WFR): {wgr_values}")
print("=" * 70)

# Размерности
NFLO = len(gas_rates)      # количество дебитов
NTHP = len(thp_bar)        # количество THP
NWFR = len(wgr_values)     # количество WFR
NGFR = len(ogr_values)     # количество GFR (у нас 1)
NALQ = len(grat_values)    # количество ALQ (у нас 1)

print(f"Всего комбинаций: {NTHP} × {NWFR} × {NGFR} × {NALQ} = {NTHP * NWFR * NGFR * NALQ}")
print(f"Каждая запись содержит {NFLO} значений BHP")
print(f"Всего расчётов: {NFLO * NTHP * NWFR * NGFR * NALQ}")
print("=" * 70)

# Массив для хранения результатов: [NTHP][NWFR][NGFR][NALQ][NFLO]
# Инициализируем массив BHP значениями 1e10 (признак ошибки)
bhp_results = np.full((NTHP, NWFR, NGFR, NALQ, NFLO), 1e10)

total_cases = NFLO * NTHP * NWFR * NGFR * NALQ
case_counter = 0

# Вложенные циклы по всем комбинациям
for i_thp in range(NTHP):
    thp = thp_bar[i_thp]
    for i_wfr in range(NWFR):
        wgr = wgr_values[i_wfr]
        for i_gfr in range(NGFR):
            ogr = ogr_values[i_gfr]  # не используется
            for i_alq in range(NALQ):
                grat = grat_values[i_alq]  # не используется
                
                # Для каждого дебита газа
                for i_flo in range(NFLO):
                    Q_gas = gas_rates[i_flo]
                    case_counter += 1
                    
                    print(f"Расчёт {case_counter}/{total_cases}: THP={thp} бар, Qgas={Q_gas} м3/сут, WGR={wgr:.6f}")
                    
                    try:
                        bhp = calculate_bhp_from_thp(
                            thp * 1e5, Q_gas, wgr, well_data_hor,
                            diameter, lyamda, Ro_water_std, sigma,
                            fluid_obj, wellhead_temp, reservoir_temp,
                            Pstd, Tstd, Ro_gas_std
                        )
                        bhp_results[i_thp, i_wfr, i_gfr, i_alq, i_flo] = bhp
                        print(f"  BHP = {bhp:.4f} бар")
                    except Exception as e:
                        print(f"  ОШИБКА: {e}")
                        bhp_results[i_thp, i_wfr, i_gfr, i_alq, i_flo] = 1e10

import pandas as pd

# После завершения всех расчётов, создаём DataFrame с подписями

# Получаем форму массива
NTHP, NWFR, NGFR, NALQ, NFLO = bhp_results.shape

# Создаём список для сбора данных
rows_data = []

# Заголовок для пояснения
print("Формирование VFP-подобной таблицы...")

for i_thp in range(NTHP):
    for i_wfr in range(NWFR):
        for i_gfr in range(NGFR):
            for i_alq in range(NALQ):
                # Получаем значения BHP для всех дебитов
                bhp_values = bhp_results[i_thp, i_wfr, i_gfr, i_alq, :]
                
                # Создаём строку с описанием параметров
                row = {
                    'THP_N': i_thp + 1,
                    'WFR_N': i_wfr + 1,
                    'GFR_N': i_gfr + 1,
                    'ALQ_N': i_alq + 1,
                    'THP_бар': thp_bar[i_thp],
                    'WGR_м3/м3': wgr_values[i_wfr],
                    'GFR_OGR': ogr_values[i_gfr],
                    'ALQ_GRAT': grat_values[i_alq]
                }
                
                # Добавляем BHP для каждого дебита газа
                for i_flo in range(NFLO):
                    row[f'BHP_FLO_{i_flo+1}_Qgas={gas_rates[i_flo]:.0f}'] = bhp_values[i_flo]
                
                rows_data.append(row)

import pandas as pd

# После завершения всех расчётов создаём DataFrame
rows_data = []

for i_thp in range(NTHP):
    for i_wfr in range(NWFR):
        for i_gfr in range(NGFR):
            for i_alq in range(NALQ):
                # Получаем BHP для всех дебитов
                bhp_values = bhp_results[i_thp, i_wfr, i_gfr, i_alq, :]
                
                # Строка с параметрами
                row = {
                    'THP_бар': thp_bar[i_thp],
                    'WGR_м3/м3': wgr_values[i_wfr],
                }
                
                # Добавляем BHP для каждого дебита
                for i_flo in range(NFLO):
                    row[f'Q={gas_rates[i_flo]:.0f}'] = round(bhp_values[i_flo], 4)
                
                rows_data.append(row)

# Создаём DataFrame
df_vfp = pd.DataFrame(rows_data)

# Просто выводим в консоль
print("\n" + "=" * 100)
print("VFP ТАБЛИЦА (BHP в барах)")
print("=" * 100)
print(df_vfp.to_string())
print("=" * 100)

# Если хочешь посмотреть в VSCode как таблицу
df_vfp


# ============================================================
# ЗАПИСЬ VFP ТАБЛИЦЫ В ФАЙЛ
# ============================================================

output_lines = []

# Строка 1: ключевое слово
output_lines.append("VFPPROD")

# Строка 2: основные данные таблицы
# номер_таблицы глубина_забоя тип_FLO тип_WFR тип_GFR тип_THP тип_ALQ единицы_измерения тип_данных
output_lines.append(
    f" {table_number} {max_TVD:.0f} GAS WGR OGR THP GRAT METRIC BHP /"
)

# Строка 3: значения FLO (дебиты газа)
flo_line = " " + " ".join([f"{q:.0f}" for q in gas_rates]) + " /"
output_lines.append(flo_line)

# Строка 4: значения THP (устьевые давления)
thp_line = " " + " ".join([f"{p:.0f}" for p in thp_bar]) + " /"
output_lines.append(thp_line)

# Строка 5: значения WFR (WGR)
wfr_line = " " + " ".join([f"{w:.6f}" for w in wgr_values]) + " /"
output_lines.append(wfr_line)

# Строка 6: значения GFR (OGR)
gfr_line = " " + " ".join([f"{g:.1f}" for g in ogr_values]) + " /"
output_lines.append(gfr_line)

# Строка 7: значения ALQ (GRAT)
alq_line = " " + " ".join([f"{a:.1f}" for a in grat_values]) + " /"
output_lines.append(alq_line)

# Строки с данными BHP
# Формат: NT NW NG NA BHP_1 BHP_2 ... BHP_NFLO /
print("\nЗапись результатов в файл...")

for i_thp in range(NTHP):
    for i_wfr in range(NWFR):
        for i_gfr in range(NGFR):
            for i_alq in range(NALQ):
                # Номера (индексация с 1)
                nt = i_thp + 1
                nw = i_wfr + 1
                ng = i_gfr + 1
                na = i_alq + 1
                
                # Значения BHP для всех дебитов газа
                bhp_values = bhp_results[i_thp, i_wfr, i_gfr, i_alq, :]
                
                # Форматируем строку
                bhp_str = " ".join([f"{bhp:.6f}" for bhp in bhp_values])
                data_line = f" {nt} {nw} {ng} {na} {bhp_str} /"
                output_lines.append(data_line)

# Сохраняем в файл
output_file = "vfp_table.inc"
with open(output_file, 'w', encoding='utf-8') as f:
    for line in output_lines:
        f.write(line + '\n')

print("\n" + "=" * 70)
print(f"VFP таблица сохранена в файл: {output_file}")
print(f"Количество строк в файле: {len(output_lines)}")
print("=" * 70)

# Выводим первые 15 строк для проверки
print("\nПервые 15 строк файла:")
print("-" * 70)
for i, line in enumerate(output_lines[:15]):
    print(line)
if len(output_lines) > 15:
    print("...")

# Выводим пример данных
print("\n" + "=" * 70)
print("ПРИМЕР ДАННЫХ В ТАБЛИЦЕ:")
print("=" * 70)
print(f"THP_1={thp_bar[0]} бар, WGR_1={wgr_values[0]:.6f}:")
for i_flo in range(NFLO):
    bhp = bhp_results[0, 0, 0, 0, i_flo]
    print(f"  Qgas={gas_rates[i_flo]:.0f} м3/сут -> BHP={bhp:.4f} бар")

ГЕНЕРАЦИЯ VFP ТАБЛИЦЫ
Таблица №1
Глубина забоя: 1200 м
Дебиты газа (FLO): [ 30000  50000 100000 200000]
Устьевые давления (THP): [ 5 30 80] бар
Водогазовый фактор (WFR): [1.e-06 1.e-05 1.e-04]
Всего комбинаций: 3 × 3 × 1 × 1 = 9
Каждая запись содержит 4 значений BHP
Всего расчётов: 36
Расчёт 1/36: THP=5 бар, Qgas=30000 м3/сут, WGR=0.000001
0.016663776169404464
0.015794825185472153
0.01615738409301296
0.016531987731059565
0.016919106015318854
0.017319230837299252
0.017732877220117595
0.018160584541560512
0.01860291782852727
0.01906046912723832
0.019533858953862622
0.0200237378305021
0.020530787911771094
0.02105572470752257
0.021599298907603464
0.022162298314865592
0.02274554989301922
0.023349921936290124
0.023976326368229085
0.024625721177424415
0.02529911299827842
0.025997559845431555
0.026722174010844504
0.027474125132980302
0.028254643447958812
0.02906502323297964
0.029906626452723618
0.03078088661983406
0.031689312880943635
0.03263349434003709
0.03361510463120924
0.03463590675308187

d:\Python_2025\Gas_liquid_flow\fluid.py:98: RuntimeWarning: invalid value encountered in scalar power
  A2 = (A0 - (A0**2 - A1**3)**0.5)**(1/3)
d:\Python_2025\Gas_liquid_flow\fluid.py:98: RuntimeWarning: invalid value encountered in scalar power
  A2 = (A0 - (A0**2 - A1**3)**0.5)**(1/3)
